Proyecto final Unidad 6 Filtros
Universidad de Antioquia

William Andres Mora Herrera
Jean Carlos Suarez Hernandez

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

# Carpeta con los .txt (relativa a la carpeta del notebook/script: Proyecto_SisFall)
CARPETA = Path("Archivos a analizar")

COLUMNAS = [
    'ADXL_X', 'ADXL_Y', 'ADXL_Z',  # Acelerómetro 1
    'ITG_X', 'ITG_Y', 'ITG_Z',     # Giroscopio
    'MMA_X', 'MMA_Y', 'MMA_Z'      # Acelerómetro 2
]

# Factores de conversión: (2 * Rango) / (2^Resolución)
FACTORES = {
    'ADXL': (2 * 16) / 2**13,    # ±16 g, 13 bits
    'ITG':  (2 * 2000) / 2**16,  # ±2000 °/s, 16 bits
    'MMA':  (2 * 8) / 2**14,     # ±8 g, 14 bits
}

def importar_y_limpiar(nombre_archivo, carpeta=CARPETA, magnitud=True, remover_gravedad=True):
    """
    Importa un .txt de SisFall, lo limpia y lo convierte a unidades físicas.
- nombre_archivo: con o sin extensión, ej. 'D04_SA01_R01' o 'D04_SA01_R01.txt'
- magnitud: True  -> devuelve 3 columnas: ADXL, ITG, MMA (magnitud del vector X,Y,Z)
                False -> devuelve las 9 columnas por componente (ADXL_X, ADXL_Y, ...)
- remover_gravedad: True -> resta el promedio a las señales de los acelerómetros
- Devuelve el DataFrame y además lo deja en memoria como df_<nombre>.
    """
    nombre = Path(nombre_archivo).stem
    ruta = Path(carpeta) / f"{nombre}.txt"

    # 1. Lectura
    df = pd.read_csv(ruta, sep=',', engine='python', header=None, names=COLUMNAS)

    # 2. Limpieza (el último dato de cada fila trae ';')
    df['MMA_Z'] = df['MMA_Z'].astype(str).str.replace(';', '').str.strip().astype(int)
    df = df.astype(int)

    # 3. Conversión de bits a valores físicos
    for prefijo, factor in FACTORES.items():
        cols = [f'{prefijo}_X', f'{prefijo}_Y', f'{prefijo}_Z']
        df[cols] = df[cols] * factor

    # 4. Remover gravedad (solo en acelerómetros)
    if remover_gravedad:
        cols_acel = ['ADXL_X', 'ADXL_Y', 'ADXL_Z', 'MMA_X', 'MMA_Y', 'MMA_Z']
        df[cols_acel] = df[cols_acel] - df[cols_acel].mean()

    # 5. Magnitud del vector resultante (opcional)
    if magnitud:
        df = pd.DataFrame({
            prefijo: np.sqrt(df[f'{prefijo}_X']**2 +
                             df[f'{prefijo}_Y']**2 +
                             df[f'{prefijo}_Z']**2)
            for prefijo in FACTORES
        })

    # 6. Guardar en memoria con el nombre df_<archivo>
    globals()[f"df_{nombre}"] = df
    print(f"✔ '{nombre}.txt' cargado → df_{nombre} ({df.shape[0]} filas, columnas: {list(df.columns)})")
    return df

In [2]:
import plotly.graph_objects as go

# 1. Factor de conversión del acelerómetro ADXL345 (según Readme: +-16g, 13 bits)
factor_g = (2 * 16) / (2**13)

# 2. Extraer columnas 0, 1 y 2 (X, Y, Z) y multiplicar por el factor
ax = datos_crudos[:, 0] * factor_g
ay = datos_crudos[:, 1] * factor_g
az = datos_crudos[:, 2] * factor_g

# 3. Calcular la magnitud de la aceleración total en cada instante t
a_total = np.sqrt(ax**2 + ay**2 + az**2)

# 4. Crear el vector de tiempo en segundos (frecuencia de muestreo = 200 Hz)

#Debido a que los archivos no nos dan los datos de los segundos, tenemos que construir el eje de tiempo a partir 
#de la cantidad de muestras y la frecuencia de muestreo. 
#La frecuencia de muestreo es de 200 Hz, es decir, 200 muestras por segundo. 
#Por lo tanto, el tiempo en segundos se puede calcular como el número de muestras dividido por la frecuencia de muestreo.
fs_muestreo = 200
tiempo = np.arange(len(a_total)) / fs_muestreo

# 5. Verificación visual en el dominio del tiempo
fig = go.Figure()
fig.add_trace(go.Scatter(x=tiempo, y=a_total, mode="lines"))
fig.update_layout(
    title="Aceleración Total a_total(t)",
    xaxis_title="Tiempo (s)",
    yaxis_title="Aceleración (g)",
    xaxis=dict(showgrid=True),
    yaxis=dict(showgrid=True),
    width=900,
    height=400,
)
fig.show()

In [3]:
import plotly.graph_objects as go

# 1. Quitar la componente constante de gravedad (0 Hz / DC)
a_sin_dc = a_total - np.mean(a_total)

# 2. Calcular la FFT para señales reales y su magnitud normalizada
N = len(a_sin_dc)
fft_compleja = np.fft.rfft(a_sin_dc)
magnitud = np.abs(fft_compleja) / N

# 3. Construir el eje X de frecuencias en Hz (sabemos que fs_muestreo = 200 Hz)
frecuencias = np.fft.rfftfreq(N, d=1 / fs_muestreo)

# 4. Graficar el espectro de frecuencias
fig = go.Figure()
fig.add_trace(go.Scatter(x=frecuencias, y=magnitud, mode="lines"))
fig.update_layout(
    title="Espectro de Frecuencias |F(f)| - Caída F03",
    xaxis_title="Frecuencia (Hz)",
    yaxis_title="Magnitud",
    xaxis=dict(range=[0, 50], showgrid=True),  # Hacemos zoom de 0 a 50 Hz donde ocurre el movimiento humano
    yaxis=dict(showgrid=True),
    width=900,
    height=400,
)
fig.show()